## Matrix Solver Reference Implementation

### Floating Point Test

In [1]:
from ieee754 import double, IEEE754

# 直接获取双精度表示
print(double(13.375))

# 使用类获取更多信息
x = 13.375
a = IEEE754(x,precision=1) # 默认双精度
print(str(a)) # 二进制表示
print(a.hex()) # 十六进制表示
print(a.json()) # 详细信息(JSON)

0 10000000010 1010110000000000000000000000000000000000000000000000
0 10000010 10101100000000000000000
('41560000', ['0100', '0001', '0101', '0110', '0000', '0000', '0000', '0000'])
{'number': Decimal('13.375'), 'edge_case': False, 'sign_bit': '1', 'exponent_bits': 8, 'mantissa_bits': 23, 'total_bits': 32, 'sign': '0', 'scale': 3, 'scaled_number': 107, 'scaled_number_in_binary': '1101011', 'unable_to_scale': False, 'bias': 127, 'bias_in_binary': '01111111', 'exponent': '10000010', 'mantissa': '10101100000000000000000', 'mantissa_base_10': Decimal('0.67187500000000000000000'), 'result': '0 10000010 10101100000000000000000', 'hexadecimal': '41560000', 'hexadecimal_parts': ['0100', '0001', '0101', '0110', '0000', '0000', '0000', '0000'], 'converted_number': Decimal('13.375'), 'error': Decimal('0.000')}


In [3]:
a, b = 6.25,13.25
a_plus_b = a+b
aa = IEEE754(a, precision=1)
bb = IEEE754(b, precision=1)
rr = IEEE754(a_plus_b, precision=1)
print(f"A        hex : {aa.hex()[0]}")
print(f"B        hex : {bb.hex()[0]}")
print(f"Expected hex : {rr.hex()[0]}")

A        hex : 40C80000
B        hex : 41540000
Expected hex : 419C0000


In [55]:
a, b, c = 122.04, 0.06152, 0.237
r = a * b + c
aa = IEEE754(a, precision=1)
bb = IEEE754(b, precision=1)
cc = IEEE754(c, precision=1)
rr = IEEE754(r, precision=1)
print(f"A        hex : {aa.hex()[0]}")
print(f"B        hex : {bb.hex()[0]}")
print(f"C        hex : {cc.hex()[0]}")
print(f"Expected hex : {rr.hex()[0]}")
print(r)
print(a/b)

A        hex : 42F4147B
B        hex : 3D7BFC65
C        hex : 3E72B021
Expected hex : 40F7D63B
7.7449008
1983.7451235370613


### LU Decomposition

In [5]:
import numpy as np

# available floating-point primitives

cycles = 0

verbose = 3


# cycle: 0
def neg(v: float) -> float:
    return -v


# cycle: 16
def fma(a: float, b: float, c: float) -> float:
    global cycles
    global verbose
    cycles += 16
    if verbose > 2:
        print(f"[fma] {a:.3f} * {b:.3f} + {c:.3f} = {a * b + c:.3f}")
    return a * b + c


# cycle: 28
def div(a: float, b: float) -> float:
    global cycles
    global verbose
    cycles += 28
    if verbose > 2:
        print(f"[div] {a:.3f} / {b:.3f} = {a / b:.3f}")
    return a / b


def lu_decomp(A: np.ndarray):
    n = A.shape[0]
    # L = np.eye(n, dtype=float)
    # U = np.zeros((n, n), dtype=float)
    L = np.random.rand(n, n)
    U = np.random.rand(n, n)
    for j in range(n):
        for i in range(n):
            if i <= j:
                U[i, j] = neg(A[i, j])
                for k in range(i):
                    # fused multiply and add
                    U[i, j] = fma(L[i, k], U[k, j], U[i, j])  # k < i
                U[i, j] = neg(U[i, j])
            else:  # i > j
                L[i, j] = neg(A[i, j])
                for k in range(j):
                    L[i, j] = fma(L[i, k], U[k, j], L[i, j])  # k < j; i > j => k < i
                L[i, j] = div(L[i, j], U[j, j])
                L[i, j] = neg(L[i, j])
    return L, U


def lu_decomp_more_hard(A: np.ndarray):
    n = A.shape[0]
    LU = np.random.rand(n, n)
    global cycles
    global verbose
    cycles = 0

    def fetch_A(i, j):
        global cycles
        global verbose
        cycles += 2
        if verbose > 2:
            print(f"fetch  A i={i} j={j} => {A[i, j]:.3f}")
        return A[i, j]

    def fetch_LU(i, j):
        global cycles
        global verbose
        cycles += 2
        if verbose > 2:
            print(f"fetch LU i={i} j={j} => {LU[i, j]:.3f}")
        return LU[i, j]

    def store_LU(i, j, v):
        global cycles
        global verbose
        cycles += 2
        if verbose > 2:
            print(f"store LU i={i} j={j} => {v:.3f}")
        LU[i, j] = v


    for idx_j in range(n):
        for idx_i in range(n):
            # module : start busy done
            f32_1 = fetch_A(idx_i, idx_j)  # 2 cycles {StateFetchA + StateWaitA}
            f32_1 = -f32_1                              # combinational [f32_1_neg] {}
            idx_m = idx_j if idx_i > idx_j else idx_i   # combinational [min_i_j]
            for idx_k in range(idx_m):
                f32_2 = fetch_LU(i=idx_i, j=idx_k)  # 2 cycles {StateFetchL + StateWaitL}
                f32_3 = fetch_LU(i=idx_k, j=idx_j)  # 2 cycles {StateFetchU + StateWaitU}
                f32_1 = fma(f32_2, f32_3, f32_1)  # 16 cycles {StateStartFMA + StateWaitFMA}
            if idx_i > idx_j:
                f32_2 = fetch_LU(i=idx_j, j=idx_j)  # 2 cycles {StateFetchDiag + StateWaitDiag}
                f32_1 = div(a=f32_1, b=f32_2)  # 28 cycles {StateStartDiv + StateWaitDiv}
            f32_1 = -f32_1                              # combinational [f32_1_neg]
            store_LU(i=idx_i, j=idx_j, v=f32_1)  # 2 cycles {StateStore + StateWaitStore}

    print(f"cycle = {cycles}")

    return LU

StateIdle
StateFetchA
StateWaitA
StateInitK
StateFetchL
StateWaitL
StateFetchU
StateWaitU
StateStartFMA
StateWaitFMA
StateCheckK
StateFetchDiag
StateWaitDiag
StateStartDiv
StateWaitDiv
StateStore
StateWaitStore
StateAdvance
StateDone

In [ ]:
cycles = 0; mat = np.random.rand(64, 64); lu_decomp(mat); print(f"64x64 cycles={cycles}")
cycles = 0; mat = np.random.rand(32, 32); lu_decomp(mat); print(f"32x32 cycles={cycles}")
cycles = 0; mat = np.random.rand(16, 16); lu_decomp(mat); print(f"16x16 cycles={cycles}")
cycles = 0; mat = np.random.rand(8, 8); lu_decomp(mat); print(f"8x8 cycles={cycles}")

64x64 cycles=1421952
32x32 cycles=180544
16x16 cycles=23200
8x8 cycles=3024


In [ ]:
import scripts.matrix2mem

mat = np.random.rand(8, 8)
# mat = np.array(
#     [
#         [4.0, 3.0, 2.0, 1.0],
#         [3.0, 5.0, 1.5, -2.0],
#         [2.0, 1.5, 6.0, 0.5],
#         [1.0, -2.0, 0.5, 7.0],
#     ],
#     dtype=np.float32,
# )
print(f"mat:{mat}")
scripts.matrix2mem.matrix_to_word_mem(mat, "tmp/lu_test_input_8x8.mem")

mat:[[0.3504328  0.35208813 0.81985258 0.20089921 0.06785319 0.44695072
  0.48418827 0.74320814]
 [0.55384571 0.21674682 0.332806   0.91189383 0.56611146 0.55156667
  0.99352573 0.92048504]
 [0.05699555 0.84861088 0.82839735 0.20314056 0.45473816 0.51801166
  0.89295197 0.91558268]
 [0.19638305 0.96227501 0.83507235 0.61745542 0.72356558 0.50704421
  0.5574652  0.24042621]
 [0.73878172 0.08009399 0.62024627 0.72338465 0.04727651 0.78315982
  0.24568511 0.66599109]
 [0.43218694 0.86138026 0.7689712  0.8254186  0.94818854 0.28711223
  0.97065568 0.20052596]
 [0.57358073 0.24448232 0.26344628 0.71631496 0.34625712 0.15636705
  0.68884915 0.96182812]
 [0.68649414 0.92614496 0.12234542 0.02389907 0.57835229 0.57772431
  0.18186042 0.6412226 ]]
Wrote 64 words to tmp\lu_test_input_8x8.mem
Matrix shape: (8, 8)
Flatten order: row-major


In [6]:
lu = lu_decomp_more_hard(mat)
uu = np.triu(lu)
ll = np.tril(lu, k=-1) + np.eye(lu.shape[0])
print(f"L:{ll}\nU:{uu}\nLU:{lu}")
print(f"L@U:{ll @ uu}")
scripts.matrix2mem.matrix_to_word_mem(lu, "tmp/lu_test_expected_8x8.mem")

fetch  A i=0 j=0 => 0.350
store LU i=0 j=0 => 0.350
fetch  A i=1 j=0 => 0.554
fetch LU i=0 j=0 => 0.350
[div] -0.554 / 0.350 = -1.580
store LU i=1 j=0 => 1.580
fetch  A i=2 j=0 => 0.057
fetch LU i=0 j=0 => 0.350
[div] -0.057 / 0.350 = -0.163
store LU i=2 j=0 => 0.163
fetch  A i=3 j=0 => 0.196
fetch LU i=0 j=0 => 0.350
[div] -0.196 / 0.350 = -0.560
store LU i=3 j=0 => 0.560
fetch  A i=4 j=0 => 0.739
fetch LU i=0 j=0 => 0.350
[div] -0.739 / 0.350 = -2.108
store LU i=4 j=0 => 2.108
fetch  A i=5 j=0 => 0.432
fetch LU i=0 j=0 => 0.350
[div] -0.432 / 0.350 = -1.233
store LU i=5 j=0 => 1.233
fetch  A i=6 j=0 => 0.574
fetch LU i=0 j=0 => 0.350
[div] -0.574 / 0.350 = -1.637
store LU i=6 j=0 => 1.637
fetch  A i=7 j=0 => 0.686
fetch LU i=0 j=0 => 0.350
[div] -0.686 / 0.350 = -1.959
store LU i=7 j=0 => 1.959
fetch  A i=0 j=1 => 0.352
store LU i=0 j=1 => 0.352
fetch  A i=1 j=1 => 0.217
fetch LU i=1 j=0 => 1.580
fetch LU i=0 j=1 => 0.352
[fma] 1.580 * 0.352 + -0.217 = 0.340
store LU i=1 j=1 => -0.34

In [2]:
import ast

source = """
def lu_kernel(n: i32):
    for j in range(n):
        for i in range(n):
            acc = -fetch_A(i, j)
            if i > j:
                acc = div(acc, fetch_LU(j, j))
            store_LU(i, j, acc)
"""

tree = ast.parse(source)
print(ast.dump(tree, indent=2))


Module(
  body=[
    FunctionDef(
      name='lu_kernel',
      args=arguments(
        args=[
          arg(
            arg='n',
            annotation=Name(id='i32', ctx=Load()))]),
      body=[
        For(
          target=Name(id='j', ctx=Store()),
          iter=Call(
            func=Name(id='range', ctx=Load()),
            args=[
              Name(id='n', ctx=Load())]),
          body=[
            For(
              target=Name(id='i', ctx=Store()),
              iter=Call(
                func=Name(id='range', ctx=Load()),
                args=[
                  Name(id='n', ctx=Load())]),
              body=[
                Assign(
                  targets=[
                    Name(id='acc', ctx=Store())],
                  value=UnaryOp(
                    op=USub(),
                    operand=Call(
                      func=Name(id='fetch_A', ctx=Load()),
                      args=[
                        Name(id='i', ctx=Load()),
                        Name(i

In [1]:
grid_width = 8
grid_height = 8


ports = [
    [
        [
            False,  # down (1,0)
            False,  # right (0,1)
            False,  # up (-1,0)
            False,  # left (0,-1)
        ]
        for j in range(grid_height)
    ]
    for i in range(grid_width)
]

dir_nxt = ((0, 1), (1, 0), (0, -1), (-1, 0))
opp_dir = (2, 3, 0, 1)

netlist_test_input_path = "data/netlist_test_data.txt"


def bitstr2bittuple(s):
    return tuple((s[i] == "1" for i in range(4)))


with open(netlist_test_input_path, "r") as input_file:
    line_n = 0
    while True:
        line = input_file.readline()
        if not line:
            break
        line_split = line.strip().split(" ")
        i = 0
        for cell in line_split:
            ports[i][line_n] = bitstr2bittuple(cell)
            i += 1
        line_n += 1
# print(ports)


def getwireports(i, j):
    return ports[i][j]


def iswire(i, j):
    return ports[i][j] != (0, 0, 0, 0)

result = [["N" for j in range(grid_height)] for i in range(grid_width)]
def bfs():
    # visited : reg
    visited = [[False for j in range(grid_height)] for i in range(grid_width)]
    queue = []
    node_idx = 0

    def add_to_queue(i, j):
        # print(f"add {(i, j)} to queue")
        visited[i][j] = True
        result[i][j] = node_idx
        for dir in range(4):
            if ports[i][j][dir]:
                queue.insert(0, (i, j, dir))

    for j in range(grid_height):
        for i in range(grid_width):
            if not visited[i][j] and iswire(i, j):
                node_idx += 1
                add_to_queue(i, j)
                # print(f"begin exploring from {(i, j)}")
                while len(queue) > 0:
                    ci, cj, dir = queue.pop()
                    dnxt = dir_nxt[dir]
                    ni, nj = ci + dnxt[0], cj + dnxt[1]
                    # print(f"at {(ci, cj)} dir {dir} => check {(ni, nj)}")
                    if (
                        not visited[ni][nj]
                        and iswire(i, j)
                        and ni >= 0
                        and nj >= 0
                        and ni < grid_width
                        and nj < grid_height
                    ):
                        if getwireports(ni, nj)[opp_dir[dir]]:
                            add_to_queue(ni, nj)
                    # if checknxt(*cur_explore):
def print_node_map():
    for j in range(grid_height):
        for i in range(grid_width):
            print(f"{result[i][j]} ",end='')
        print("")
bfs()
print_node_map()

1 1 N N 2 2 2 3 
4 1 1 2 2 5 N 3 
4 4 4 N 5 5 N 3 
4 4 6 N 5 3 3 3 
4 7 6 6 5 N 3 N 
4 7 7 6 5 N 3 N 
4 5 5 5 5 5 3 4 
4 4 4 4 4 4 4 4 


In [ ]:
def bfs_hard_1():
    reset()
    node_idx = 0
    for j in range(grid_height):
        for i in range(grid_width):
            if check_1(i,j): # visited implemented as ram, wire have to check port ram (2 cycle)
                node_idx += 1 # combinational
                add_to_queue(i, j) # queue implemented as ram + reg idx (not optimized -- 4 cycles)
                while queue_len > 0:
                    target_dir,ni,nj,real_dir = pop_queue() # 4 cycles
                    if check_2(target_dir,ni,nj,real_dir): # visited / wire stuff (2 cycles)
                        add_to_queue(ni, nj) # 4 cycles

CellVisStore


ComponentStore
  - ports (from RAM 1 & RAM 2) 4b per cell

- Substore for netlist generation
  - visited (internal) 1b per cell
  - queue